## Goal:

Build simple benchmark models to predict `subrogation` (0/1), save the trained model artifacts, and append each model's validation performance to a single JSON experiment log.

This notebook starts with baseline Logistic Regression and XGBoost models. The JSON log will allow us to compare future fine-tuning runs against these baselines.

## 1. Imports and project paths

In [1]:
import json
import joblib
import pandas as pd

from pathlib import Path
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier


In [2]:
# If this notebook is inside a notebooks/ folder, this points to the project root.
# Otherwise, it uses the current working directory as the project root.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "train_features.csv"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
METRICS_PATH = REPORTS_DIR / "model_metrics.json"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH)
print("Metrics path:", METRICS_PATH)

Project root: /Users/eugene/Desktop/Emory/Projects/subrogation-project
Data path: /Users/eugene/Desktop/Emory/Projects/subrogation-project/data/processed/train_features.csv
Metrics path: /Users/eugene/Desktop/Emory/Projects/subrogation-project/reports/model_metrics.json


In [3]:
# df = pd.read_csv("/Users/eugene/Desktop/Emory/Projects/subrogation-project/data/processed/train_features.csv")
# df.head()

## 2. Load processed training data

In [4]:
df = pd.read_csv(DATA_PATH)
df.head()

,subrogation,email_or_tel_available,safety_rating,high_education_ind,address_change_ind,accident_site,witness_present_ind,liab_prct,channel,policy_report_filed_ind,vehicle_made_year,vehicle_weight,accident_type,in_network_bodyshop,vehicle_mileage,claim_quarter,is_holiday_season,log_claim_est_payout,log_vehicle_price,log_annual_income
0,1,0,75.0,1,1,Parking Area,1,31.0,Broker,1,2021.0,21620.79697,multi_vehicle_clear,no,75421.0,4,1,8.077087,9.697270,11.169970
1,0,1,94.0,1,1,Parking Area,0,34.0,Phone,1,2025.0,10840.58520,multi_vehicle_clear,yes,31988.0,2,0,7.200067,10.437164,11.286326
2,0,1,76.0,1,1,Unknown,0,39.0,Broker,1,2022.0,24318.12282,multi_vehicle_unclear,yes,60876.0,2,0,8.172179,9.615872,10.634123
3,1,1,54.0,1,1,Unknown,0,32.0,Phone,1,2025.0,36958.26656,multi_vehicle_unclear,yes,152772.0,1,0,7.319163,9.740113,10.647803
4,0,1,54.0,1,1,Highway/Intersection,0,28.0,Online,0,2021.0,11779.17422,multi_vehicle_clear,yes,41151.0,1,0,8.533387,10.748212,10.762297


## 3. Train/validation split and encoding

For this baseline notebook, we keep the simple one-hot encoding approach. The aligned feature columns are saved with each model so future inference uses the same feature order.

In [5]:
X = df.drop(columns=["subrogation"])
y = df["subrogation"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# One-hot encode categorical variables
X_train = pd.get_dummies(X_train, drop_first=True)
X_val = pd.get_dummies(X_val, drop_first=True)

# Align validation columns to match training columns
X_train, X_val = X_train.align(X_val, join="left", axis=1, fill_value=0)

feature_columns = X_train.columns.tolist()

print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Target distribution in training:")
print(y_train.value_counts(normalize=True))

Training shape: (14399, 23)
Validation shape: (3600, 23)
Target distribution in training:
subrogation
0    0.771373
1    0.228627
Name: proportion, dtype: float64


## 4. Reusable evaluation and experiment logging functions

These functions:
- evaluate a fitted model,
- save the model as a `.pkl` file,
- append model metrics to one JSON file,
- store run metadata such as timestamp, parameters, model path, and feature columns.


In [6]:
def make_json_safe(value):
    """
    Convert values that may not be JSON serializable into safe Python objects.
    """
    try:
        json.dumps(value)
        return value
    except TypeError:
        return str(value)


def evaluate_model(model, X_val, y_val):
    """
    Evaluate a fitted binary classification model on validation data.
    Return only the key metrics needed for experiment tracking.
    """
    y_pred = model.predict(X_val)

    metrics = {
        "accuracy": round(accuracy_score(y_val, y_pred), 4),
        "precision": round(precision_score(y_val, y_pred, zero_division=0), 4),
        "recall": round(recall_score(y_val, y_pred, zero_division=0), 4),
        "f1_score": round(f1_score(y_val, y_pred, zero_division=0), 4),
    }

    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_val)[:, 1]
        metrics["roc_auc"] = round(roc_auc_score(y_val, y_proba), 4)
    else:
        metrics["roc_auc"] = None

    return metrics


def save_model_and_append_metrics(
    model,
    model_name,
    model_family,
    X_val,
    y_val,
    metrics_path=METRICS_PATH,
    models_dir=MODELS_DIR,
    notes=None
):
    """
    Save a trained model and append key validation metrics to a single JSON file.
    """
    timestamp = datetime.now().strftime("%Y-%m-%d_%H%M%S")
    run_id = f"{timestamp}_{model_name}"

    model_path = models_dir / f"{run_id}.pkl"

    # Save trained model artifact
    joblib.dump(model, model_path)

    # Evaluate model
    metrics = evaluate_model(model, X_val, y_val)

    # Keep the experiment log simple and readable
    run_record = {
        "run_id": run_id,
        "run_date": datetime.now().isoformat(timespec="seconds"),
        "model_name": model_name,
        "model_family": model_family,
        "model_path": str(model_path),
        "metrics": metrics,
        "notes": notes
    }

    # Load existing experiment log if it exists
    if metrics_path.exists():
        with open(metrics_path, "r") as file:
            records = json.load(file)

        # Handles the case where an older metrics file was stored as a dict
        if isinstance(records, dict):
            records = [records]
    else:
        records = []

    records.append(run_record)

    with open(metrics_path, "w") as file:
        json.dump(records, file, indent=4)

    print(f"Saved model: {model_path}")
    print(f"Appended metrics to: {metrics_path}")

    return run_record


## 5. Baseline Model 1: Logistic Regression

This is the main interpretable benchmark. Because subrogation is likely imbalanced, we use `class_weight="balanced"` and track precision, recall, F1, and ROC-AUC.



In [7]:
log_reg = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

log_reg.fit(X_train, y_train)

log_reg_record = save_model_and_append_metrics(
    model=log_reg,
    model_name="logistic_regression_baseline_v1",
    model_family="LogisticRegression",
    X_val=X_val,
    y_val=y_val,
    notes="Baseline logistic regression with class_weight='balanced'."
)

log_reg_record["metrics"]


Saved model: /Users/eugene/Desktop/Emory/Projects/subrogation-project/models/2026-04-27_140757_logistic_regression_baseline_v1.pkl
Appended metrics to: /Users/eugene/Desktop/Emory/Projects/subrogation-project/reports/model_metrics.json


/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


{'accuracy': 0.7344,
 'precision': 0.4511,
 'recall': 0.7448,
 'f1_score': 0.5619,
 'roc_auc': 0.8144}

## 6. Baseline Model 2: XGBoost

This provides a stronger tree-based baseline that can capture non-linear patterns.


In [8]:
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

xgb.fit(X_train, y_train)

xgb_record = save_model_and_append_metrics(
    model=xgb,
    model_name="xgboost_baseline_v1",
    model_family="XGBClassifier",
    X_val=X_val,
    y_val=y_val,
    notes="Baseline XGBoost model before hyperparameter tuning."
)

xgb_record["metrics"]


Saved model: /Users/eugene/Desktop/Emory/Projects/subrogation-project/models/2026-04-27_140758_xgboost_baseline_v1.pkl
Appended metrics to: /Users/eugene/Desktop/Emory/Projects/subrogation-project/reports/model_metrics.json


{'accuracy': 0.8069,
 'precision': 0.6103,
 'recall': 0.4301,
 'f1_score': 0.5046,
 'roc_auc': 0.82}

## 7. Compare logged model runs

This reads the JSON experiment log and converts it into a table for easier comparison.


In [9]:
with open(METRICS_PATH, "r") as file:
    records = json.load(file)

metrics_df = pd.json_normalize(records)

comparison_cols = [
    "run_id",
    "model_name",
    "model_family",
    "metrics.accuracy",
    "metrics.precision",
    "metrics.recall",
    "metrics.f1_score",
    "metrics.roc_auc",
    "model_path",
    "notes"
]

metrics_df[comparison_cols].sort_values(
    by="metrics.f1_score",
    ascending=False
).head(20)


,run_id,model_name,model_family,metrics.accuracy,metrics.precision,metrics.recall,metrics.f1_score,metrics.roc_auc,model_path,notes
0,2026-04-27_140727_logistic_regression_baseline_v1,logistic_regression_baseline_v1,LogisticRegression,0.7344,0.4511,0.7448,0.5619,0.8144,/Users/eugene/Desktop/Emory/Projects/subrogati...,Baseline logistic regression with class_weight...
2,2026-04-27_140757_logistic_regression_baseline_v1,logistic_regression_baseline_v1,LogisticRegression,0.7344,0.4511,0.7448,0.5619,0.8144,/Users/eugene/Desktop/Emory/Projects/subrogati...,Baseline logistic regression with class_weight...
1,2026-04-27_140741_xgboost_baseline_v1,xgboost_baseline_v1,XGBClassifier,0.8069,0.6103,0.4301,0.5046,0.8200,/Users/eugene/Desktop/Emory/Projects/subrogati...,Baseline XGBoost model before hyperparameter t...
3,2026-04-27_140758_xgboost_baseline_v1,xgboost_baseline_v1,XGBClassifier,0.8069,0.6103,0.4301,0.5046,0.8200,/Users/eugene/Desktop/Emory/Projects/subrogati...,Baseline XGBoost model before hyperparameter t...


## 8. Recommended model selection logic

For this project, accuracy should not be the main metric for choosing the best model. In a subrogation problem, the target class is likely imbalanced, meaning there may be far fewer claims with subrogation potential than claims without it. Because of that, a model can look accurate while still missing many of the cases we actually care about.

The model should be selected based on the business goal:

Use recall when the priority is to identify as many potential recovery opportunities as possible.
Use precision when false positives would create too much unnecessary manual review work.
Use F1-score when we want a reasonable balance between precision and recall.
Use ROC-AUC to understand how well the model separates likely subrogation claims from non-subrogation claims overall.

For the baseline phase, F1-score is a useful headline metric because it balances precision and recall. However, precision and recall should still be reviewed together before deciding which model is actually better.